## 0. Setup 

In [ ]:
import os, sys, glob, gc, json, warnings, subprocess
warnings.filterwarnings("ignore")

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import duckdb
except ImportError:
    pipq("duckdb"); import duckdb
try:
    import lightgbm as lgb
except ImportError:
    pipq("lightgbm"); import lightgbm as lgb
try:
    import xgboost as xgb
except ImportError:
    pipq("xgboost"); import xgboost as xgb
try:
    import catboost as cb
except ImportError:
    pipq("catboost"); import catboost as cb

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
print("duckdb", duckdb.__version__, "| lgb", lgb.__version__, "| xgb", xgb.__version__, "| cb", cb.__version__)


## 1. Locate data 

In [ ]:
DATA_DIR = None
for cand in ["/kaggle/input", "../input", "./public", ".", "./data"]:
    hits = glob.glob(os.path.join(cand, "**", "train_labels.csv"), recursive=True)
    if hits:
        DATA_DIR = os.path.dirname(hits[0]); break
assert DATA_DIR, "Could not find train_labels.csv — set DATA_DIR manually."
print("DATA_DIR =", DATA_DIR)

def find_files(folder_name):
    out = []
    for ext in ("parquet", "csv", "csv.gz"):
        out += glob.glob(os.path.join(DATA_DIR, folder_name, "**", f"*.{ext}"), recursive=True)
        out += glob.glob(os.path.join(DATA_DIR, folder_name, f"*.{ext}"))
    return sorted(set(out))

TRX_FILES  = find_files("transactions")
BAL_FILES  = find_files("dayend_balance")
KYC_FILE   = glob.glob(os.path.join(DATA_DIR, "**", "kyc.*"), recursive=True)[0]
print(f"transactions: {len(TRX_FILES)} files | balance: {len(BAL_FILES)} files | kyc: {KYC_FILE}")

def duck_src(files):
    files_sql = "[" + ",".join(f"'{f}'" for f in files) + "]"
    if files[0].endswith(".parquet"):
        return f"read_parquet({files_sql}, union_by_name=true)"
    return f"read_csv_auto({files_sql}, union_by_name=true)"

TRX_SRC, BAL_SRC = duck_src(TRX_FILES), duck_src(BAL_FILES)
KYC_SRC = duck_src([KYC_FILE])

# DuckDB tuned for larger-than-RAM aggregation (Stage 1 requirement).
# Key OOM-avoidance settings (per duckdb.org workload-tuning guide):
#   - memory_limit well below physical RAM so the OS/pandas have headroom
#   - preserve_insertion_order=false (lets DuckDB stream/spill aggressively)
#   - fewer threads = less concurrent hash-table memory
#   - spill directory on the biggest writable disk
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except Exception:
    ram_gb = 13.0
MEM_LIMIT = max(2, int(ram_gb * 0.55))  # ~7GB on a 13GB Kaggle CPU instance

SPILL = "/kaggle/working/duck_spill" if os.path.isdir("/kaggle/working") else "/tmp/duck_spill"
os.makedirs(SPILL, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET temp_directory='{SPILL}'")
con.execute(f"SET memory_limit='{MEM_LIMIT}GB'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads TO 2")
print(f"DuckDB: memory_limit={MEM_LIMIT}GB, spill={SPILL}")


## 2. Schema detection 

In [ ]:
def cols_of(src):
    return [c[0] for c in con.execute(f"DESCRIBE SELECT * FROM {src} LIMIT 0").fetchall()]

def pick(cols, *cands):
    low = {c.lower(): c for c in cols}
    for cand in cands:
        if cand.lower() in low: return low[cand.lower()]
    for cand in cands:  # substring fallback
        for c in cols:
            if cand.lower() in c.lower(): return c
    raise KeyError(f"None of {cands} found in {cols}")

tc, bc, kc = cols_of(TRX_SRC), cols_of(BAL_SRC), cols_of(KYC_SRC)
print("TRX cols:", tc, "\nBAL cols:", bc, "\nKYC cols:", kc)

COLMAP = dict(
    trx_dt   = pick(tc, "TRX_DATETIME", "datetime", "date"),
    trx_src  = pick(tc, "SRC_ACCOUNT", "src"),
    trx_dst  = pick(tc, "DST_ACCOUNT", "dst"),
    trx_type = pick(tc, "TRX_TYPE", "type"),
    trx_amt  = pick(tc, "TRX_AMT", "amount", "amt"),
    bal_id   = pick(bc, "ACCOUNT_ID", "account"),
    bal_dt   = pick(bc, "DATE", "dt"),
    bal_val  = pick(bc, "AVAILABLE_BALANCE", "balance"),
    kyc_id   = pick(kc, "ACCOUNT_ID", "account"),
    kyc_type = pick(kc, "ACCOUNT_TYPE", "type"),
    kyc_open = pick(kc, "ACCOUNT_OPEN_DATE", "open"),
    kyc_gen  = pick(kc, "GENDER", "gender"),
    kyc_reg  = pick(kc, "REGION", "region"),
)
print(json.dumps(COLMAP, indent=2))

REF_DATE = con.execute(
    f"SELECT MAX(CAST(\"{COLMAP['trx_dt']}\" AS TIMESTAMP)) FROM {TRX_SRC}"
).fetchone()[0]
print("Reference (snapshot) date:", REF_DATE)


## 3a. Distinct transaction types (small query) 

In [ ]:
ttypes = [r[0] for r in con.execute(
    f"SELECT UPPER(CAST(\"{COLMAP['trx_type']}\" AS VARCHAR)) t, COUNT(*) c FROM {TRX_SRC} GROUP BY 1 ORDER BY c DESC LIMIT 10"
).fetchall()]
print("TRX types:", ttypes)
def safe(t): return "".join(ch if ch.isalnum() else "_" for ch in t.lower())


## 3b. Sent-side features (dense windows + fractional recency) 

In [ ]:
d = COLMAP
REF = str(REF_DATE)
WINDOWS = [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90]

T_SENT = f'''(
  SELECT "{d['trx_src']}" AS acct,
         CAST("{d['trx_dt']}" AS TIMESTAMP) AS ts,
         CAST("{d['trx_amt']}" AS DOUBLE)   AS amt,
         UPPER(CAST("{d['trx_type']}" AS VARCHAR)) AS typ,
         "{d['trx_dst']}" AS cpty
  FROM {TRX_SRC}
)'''

q1 = f'''
SELECT acct,
  COUNT(*)                          AS s_cnt_all,
  SUM(amt)                          AS s_amt_all,
  AVG(amt)                          AS s_amt_mean,
  STDDEV(amt)                       AS s_amt_std,
  MAX(amt)                          AS s_amt_max,
  approx_quantile(amt, 0.5)         AS s_amt_median,
  approx_count_distinct(cpty)       AS s_n_counterparties,
  approx_count_distinct(CAST(ts AS DATE)) AS s_active_days,
  date_diff('second', MAX(ts), TIMESTAMP '{REF}')/86400.0 AS s_recency_days,
  date_diff('second', MIN(ts), TIMESTAMP '{REF}')/86400.0 AS s_first_trx_days,
  AVG(CASE WHEN dayofweek(ts) IN (0,6) THEN 1 ELSE 0 END) AS s_weekend_ratio,
  AVG(hour(ts))                     AS s_hour_mean,
  -- monthly counts for decay trend (Jan / Feb / Mar)
  COUNT(*) FILTER (month(ts)=1) AS s_cnt_m1,
  COUNT(*) FILTER (month(ts)=2) AS s_cnt_m2,
  COUNT(*) FILTER (month(ts)=3) AS s_cnt_m3
FROM {T_SENT} t GROUP BY acct
'''
sent = con.execute(q1).df(); print("pass 1:", sent.shape); gc.collect()

win_sql = ",\n".join(
    f"  COUNT(*) FILTER (ts >= TIMESTAMP '{REF}' - INTERVAL {w} DAY) AS s_cnt_{w}d,\n"
    f"  COALESCE(SUM(amt) FILTER (ts >= TIMESTAMP '{REF}' - INTERVAL {w} DAY),0) AS s_amt_{w}d"
    for w in WINDOWS
)
q2 = f'''
SELECT acct,
{win_sql},
  COUNT(*) FILTER (ts <  TIMESTAMP '{REF}' - INTERVAL 30 DAY
               AND ts >= TIMESTAMP '{REF}' - INTERVAL 60 DAY) AS s_cnt_prev30
FROM {T_SENT} t GROUP BY acct
'''
sent = sent.merge(con.execute(q2).df(), on="acct", how="outer"); print("pass 2:", sent.shape); gc.collect()

type_sql = ",\n".join(f"  COUNT(*) FILTER (typ = '{t}') AS s_cnt_type_{safe(t)}" for t in ttypes)
q3 = f"SELECT acct,\n{type_sql}\nFROM {T_SENT} t GROUP BY acct"
sent = sent.merge(con.execute(q3).df(), on="acct", how="outer")
print("sent features:", sent.shape); gc.collect()


## 3c. Received-side features (dense windows + fractional recency) 

In [ ]:
T_RECV = f'''(
  SELECT "{d['trx_dst']}" AS acct,
         CAST("{d['trx_dt']}" AS TIMESTAMP) AS ts,
         CAST("{d['trx_amt']}" AS DOUBLE)   AS amt,
         "{d['trx_src']}" AS cpty
  FROM {TRX_SRC}
)'''
q_r1 = f'''
SELECT acct,
  COUNT(*) AS r_cnt_all, SUM(amt) AS r_amt_all, AVG(amt) AS r_amt_mean, STDDEV(amt) AS r_amt_std,
  approx_count_distinct(cpty) AS r_n_counterparties,
  date_diff('second', MAX(ts), TIMESTAMP '{REF}')/86400.0 AS r_recency_days
FROM {T_RECV} t GROUP BY acct
'''
recv = con.execute(q_r1).df(); print("recv pass 1:", recv.shape); gc.collect()
rwin_sql = ",\n".join(
    f"  COUNT(*) FILTER (ts >= TIMESTAMP '{REF}' - INTERVAL {w} DAY) AS r_cnt_{w}d"
    for w in WINDOWS)
recv = recv.merge(con.execute(f"SELECT acct,\n{rwin_sql}\nFROM {T_RECV} t GROUP BY acct").df(),
                  on="acct", how="outer")
print("recv features:", recv.shape); gc.collect()


## 3d. Balance dynamics + movement recency

In [ ]:
 BAL_DATE = str(pd.Timestamp(REF_DATE).date())
q_bal = f'''
WITH b AS (
  SELECT "{d['bal_id']}" AS acct, CAST("{d['bal_dt']}" AS DATE) AS dt,
         CAST("{d['bal_val']}" AS DOUBLE) AS bal FROM {BAL_SRC}
),
bl AS (SELECT acct, dt, bal, lag(bal) OVER (PARTITION BY acct ORDER BY dt) AS prev_bal FROM b)
SELECT acct,
  AVG(bal) AS bal_mean, STDDEV(bal) AS bal_std, MIN(bal) AS bal_min, MAX(bal) AS bal_max,
  approx_quantile(bal, 0.5) AS bal_median, arg_max(bal, dt) AS bal_last,
  regr_slope(bal, date_diff('day', DATE '1970-01-01', dt)) AS bal_slope,
  AVG(CASE WHEN bal <= 0 THEN 1 ELSE 0 END) AS bal_zero_ratio,
  approx_count_distinct(bal) AS bal_n_distinct,
  AVG(bal) FILTER (dt >= DATE '{BAL_DATE}' - INTERVAL 30 DAY) AS bal_mean_30d,
  STDDEV(bal) FILTER (dt >= DATE '{BAL_DATE}' - INTERVAL 30 DAY) AS bal_std_30d,
  date_diff('day', MAX(dt) FILTER (prev_bal IS NULL OR bal <> prev_bal), DATE '{BAL_DATE}') AS days_since_bal_change,
  date_diff('day', MAX(dt) FILTER (bal > 0), DATE '{BAL_DATE}') AS days_since_pos_bal,
  COUNT(*) FILTER (prev_bal IS NOT NULL AND bal <> prev_bal) AS bal_n_changes
FROM bl GROUP BY acct
'''
bal = con.execute(q_bal).df(); print("balance features:", bal.shape); gc.collect()


## 3f. Inter-transaction-gap survival features (the main new bet)

In [ ]:
# Computed at active-DAY granularity (<=90 days/account) so the window function
# runs over a much smaller set than the 200M raw rows.
q_surv = f'''
WITH ev AS (
  SELECT "{d['trx_src']}" AS acct, CAST("{d['trx_dt']}" AS DATE) AS dt
  FROM {TRX_SRC} GROUP BY 1, 2
),
g AS (
  SELECT acct, dt,
    date_diff('day', lag(dt) OVER (PARTITION BY acct ORDER BY dt), dt) AS gap
  FROM ev
)
SELECT acct,
  MAX(gap)                  AS s_max_gap,
  AVG(gap)                  AS s_avg_gap,
  STDDEV(gap)               AS s_std_gap,
  arg_max(gap, dt)          AS s_last_gap,
  approx_quantile(gap, 0.9) AS s_p90_gap
FROM g GROUP BY acct
'''
surv = con.execute(q_surv).df()
print("survival features:", surv.shape); gc.collect()


## 3e. KYC + assemble (adds survival-derived features) 

In [ ]:
kyc = con.execute(f'''
SELECT "{d['kyc_id']}" AS acct,
       CAST("{d['kyc_type']}" AS VARCHAR) AS account_type,
       CAST("{d['kyc_gen']}"  AS VARCHAR) AS gender,
       CAST("{d['kyc_reg']}"  AS VARCHAR) AS region,
       date_diff('day', CAST("{d['kyc_open']}" AS DATE), DATE '{pd.Timestamp(REF_DATE).date()}') AS account_age_days
FROM {KYC_SRC}
''').df()

train_labels = pd.read_csv(glob.glob(os.path.join(DATA_DIR, "**", "train_labels.csv"), recursive=True)[0])
test_ids     = pd.read_csv(glob.glob(os.path.join(DATA_DIR, "**", "test.csv"), recursive=True)[0])
id_col      = pick(list(train_labels.columns), "ACCOUNT_ID", "id", "account")
target_col  = [c for c in train_labels.columns if c != id_col][0]
test_id_col = pick(list(test_ids.columns), "ACCOUNT_ID", "id", "account")
print(f"id={id_col} target={target_col} | churn rate = {train_labels[target_col].mean():.4f}")

EPS = 1e-9
def assemble(ids_df, idc):
    X = ids_df[[idc]].rename(columns={idc: "acct"})
    for tbl in (kyc, sent, recv, bal, surv):
        X = X.merge(tbl, on="acct", how="left")

    cnt_cols = [c for c in X.columns if c.startswith(("s_cnt", "r_cnt"))]
    X[cnt_cols] = X[cnt_cols].fillna(0)
    for c in ["s_recency_days", "r_recency_days", "s_first_trx_days",
              "days_since_pos_bal", "days_since_bal_change"]:
        if c in X: X[c] = X[c].fillna(9999)
    X["any_recency_days"] = X[["s_recency_days", "r_recency_days"]].min(axis=1)

    for w in WINDOWS:
        s = X[f"s_cnt_{w}d"]; r = X[f"r_cnt_{w}d"]
        X[f"any_cnt_{w}d"] = s + r
        X[f"any_inactive_{w}d"]  = (X[f"any_cnt_{w}d"] == 0).astype(int)
        X[f"sent_inactive_{w}d"] = (s == 0).astype(int)

    for w in [7, 14, 30, 60, 90]:
        X[f"pois_p0_sent_{w}"] = np.exp(-(X[f"s_cnt_{w}d"] / w) * 30)
        X[f"pois_p0_any_{w}"]  = np.exp(-(X[f"any_cnt_{w}d"] / w) * 30)

    span = X["s_first_trx_days"].clip(lower=1)
    lam_hist = X["s_cnt_all"] / span
    X["exp_gap_sent"]   = span / X["s_cnt_all"].replace(0, np.nan)
    X["gap_ratio_sent"] = X["s_recency_days"] / (X["exp_gap_sent"] + EPS)
    X["silence_prob_sent"] = np.exp(-lam_hist * X["s_recency_days"])
    X["hist_rate_sent"] = lam_hist

    # ---- NEW survival-derived features ----
    for c in ["s_max_gap", "s_avg_gap", "s_std_gap", "s_last_gap", "s_p90_gap"]:
        if c in X: X[c] = X[c].fillna(0)
    X["s_recency_vs_maxgap"] = X["s_recency_days"] / (X["s_max_gap"] + 1)
    X["s_recency_vs_avggap"] = X["s_recency_days"] / (X["s_avg_gap"] + 1)
    X["s_recency_vs_p90gap"] = X["s_recency_days"] / (X["s_p90_gap"] + 1)
    X["s_broke_pattern"] = ((X["s_active_days"] >= 2) &
                            (X["s_recency_days"] > X["s_max_gap"])).astype(int)
    X["s_gap_zscore"] = (X["s_recency_days"] - X["s_avg_gap"]) / (X["s_std_gap"] + 1)

    X["decay_m3_vs_m1"]  = (X["s_cnt_m3"] + 1) / (X["s_cnt_m1"] + 1)
    X["decay_m3_vs_m2"]  = (X["s_cnt_m3"] + 1) / (X["s_cnt_m2"] + 1)
    X["trend_30_vs_prev30"] = (X["s_cnt_30d"] + 1) / (X["s_cnt_prev30"] + 1)
    X["rate_last30"] = X["s_cnt_30d"] / 30.0
    X["rate_30_90"]  = (X["s_cnt_90d"] - X["s_cnt_30d"]).clip(lower=0) / 60.0
    X["rate_accel"]  = (X["rate_last30"] + EPS) / (X["rate_30_90"] + EPS)

    X["sent_recv_ratio"]  = X["s_cnt_all"] / X["r_cnt_all"].replace(0, np.nan)
    X["bal_last_vs_mean"] = X["bal_last"] / X["bal_mean"].replace(0, np.nan)
    X["amt_per_active_day"] = X["s_amt_all"] / X["s_active_days"].replace(0, np.nan)
    for t in ttypes:
        c = f"s_cnt_type_{safe(t)}"
        X[f"ratio_type_{safe(t)}"] = X[c] / X["s_cnt_all"].replace(0, np.nan)

    for c in ["gap_ratio_sent", "rate_accel", "decay_m3_vs_m1", "decay_m3_vs_m2",
              "trend_30_vs_prev30", "sent_recv_ratio", "bal_last_vs_mean",
              "amt_per_active_day", "s_recency_vs_maxgap", "s_recency_vs_avggap",
              "s_recency_vs_p90gap", "s_gap_zscore"]:
        if c in X:
            X[c] = X[c].replace([np.inf, -np.inf], np.nan).clip(upper=1e4)

    # NEW: transaction regularity + time-to-next-expected (periodicity) 
    X["gap_cv"] = X["s_std_gap"] / (X["s_avg_gap"] + 1)                 # low = regular/periodic
    X["days_to_next_expected"] = (X["s_avg_gap"] - X["s_recency_days"]) # >0 due later, <0 overdue
    X["next_due_within_30"] = (X["days_to_next_expected"].between(-1e9, 30)).astype(int)
    X["regular_and_recent"] = ((X["gap_cv"] < 1.0) & (X["s_recency_days"] <= X["s_avg_gap"] + 1)).astype(int)

    # artifact probe feature: numeric suffix of the account id 
    X["id_numeric"] = pd.to_numeric(
        X["acct"].astype(str).str.extract(r"(\d+)$")[0], errors="coerce").fillna(-1)

    for c in ["gap_cv", "days_to_next_expected"]:
        X[c] = X[c].replace([np.inf, -np.inf], np.nan).clip(lower=-1e4, upper=1e4)

    X["never_sent"] = (X["s_cnt_all"] == 0).astype(int)
    X["never_recv"] = (X["r_cnt_all"] == 0).astype(int)
    for c in list(X.columns):
        if c.startswith(("s_amt", "r_amt", "bal_")) and X[c].dtype.kind == "f":
            X[f"log1p_{c}"] = np.sign(X[c]) * np.log1p(np.abs(X[c]))
    return X

X_train = assemble(train_labels, id_col)
X_test  = assemble(test_ids, test_id_col)
y = train_labels[target_col].values

CAT_COLS = ["account_type", "gender", "region"]
for c in CAT_COLS:
    X_train[c] = X_train[c].astype("category")
    X_test[c]  = pd.Categorical(X_test[c], categories=X_train[c].cat.categories)

FEATS = [c for c in X_train.columns if c != "acct"]
print(f"Feature matrix: train {X_train.shape}, test {X_test.shape}, {len(FEATS)} features")
del sent, recv, bal, surv; gc.collect()


## 3g. Single-feature forecasting power 

In [ ]:
from sklearn.metrics import roc_auc_score
def sfauc(v):
    v = pd.Series(v).fillna(0).values
    a = roc_auc_score(y, v); return max(a, 1 - a)

cands = (["s_recency_days", "any_recency_days", "r_recency_days",
          "days_since_bal_change", "days_since_pos_bal",
          "silence_prob_sent", "gap_ratio_sent", "hist_rate_sent",
          "decay_m3_vs_m1", "rate_accel",
          "s_recency_vs_maxgap", "s_recency_vs_avggap", "s_recency_vs_p90gap",
          "s_broke_pattern", "s_gap_zscore", "s_max_gap"]
         + [f"pois_p0_sent_{w}" for w in [7,14,30,60,90]]
         + [f"pois_p0_any_{w}"  for w in [7,14,30,60,90]]
         + [f"s_cnt_{w}d" for w in [7,14,30,90]]
         + ["gap_cv", "days_to_next_expected", "next_due_within_30", "regular_and_recent"])
diag = (pd.DataFrame([(c, sfauc(X_train[c].values)) for c in cands if c in X_train],
                     columns=["signal", "single_feature_AUC"])
        .sort_values("single_feature_AUC", ascending=False).reset_index(drop=True))
print(diag.to_string(index=False))
print(f"\nTop single signal: {diag.iloc[0]['signal']} = {diag.iloc[0]['single_feature_AUC']:.5f}")


## 3h. Artifact / leakage probes 

In [ ]:
probes = []
probes.append(("id_numeric_suffix", sfauc(X_train["id_numeric"].values)))
probes.append(("train_row_order",   sfauc(np.arange(len(X_train), dtype=float))))
for c in ["account_age_days"]:
    if c in X_train: probes.append((c, sfauc(X_train[c].fillna(0).values)))
adf = (pd.DataFrame(probes, columns=["probe", "single_feature_AUC"])
       .sort_values("single_feature_AUC", ascending=False))
print(adf.to_string(index=False))
hi = adf["single_feature_AUC"].max()
print(f"\nMax artifact AUC = {hi:.4f}")
print("  > ~0.55  -> exploitable leak; KEEP that feature, it can move you toward 0.99")
print("  ~ 0.50   -> no artifact; score is bounded by behavioral signal, 0.99 may be out of reach")


## 4. Modeling — shared-fold OOF, then a fair ensemble test

## 4a. GPU detection + CV / model setup

In [ ]:
SEEDS = [44, 43, 45, 46, 47]   # 5-seed ensemble
spw = (y == 0).sum() / max((y == 1).sum(), 1)
print(f"scale_pos_weight = {spw:.2f}")

# GPU detection (requires a GPU-enabled Kaggle session: Settings -> Accelerator) 
import subprocess
try:
    HAS_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    HAS_GPU = False

# LightGBM: GPU support depends on how the wheel was compiled
# ("cuda" = CUDA build, "gpu" = OpenCL build). Probe on a tiny sample, fall back to CPU.
LGB_DEVICE = "cpu"
if HAS_GPU:
    rs = np.random.RandomState(0).choice(len(X_train), size=min(5000, len(X_train)), replace=False)
    for dev in ("cuda", "gpu"):
        try:
            lgb.LGBMClassifier(device=dev, n_estimators=5, num_leaves=15,
                               max_bin=255, verbosity=-1).fit(X_train.iloc[rs][FEATS], y[rs])
            LGB_DEVICE = dev
            break
        except Exception:
            pass

print(f"GPU detected: {HAS_GPU} | LightGBM device: {LGB_DEVICE} | "
      f"CatBoost: {'GPU' if HAS_GPU else 'CPU'}")
if HAS_GPU and LGB_DEVICE == "cpu":
    print("note: this LightGBM wheel has no GPU support -> LightGBM stays on CPU "
          "(XGBoost + CatBoost on GPU still cut most of the runtime).")

def lgb_params(seed):
    p = dict(
        objective="binary", metric="auc", learning_rate=0.02,
        num_leaves=128, max_depth=-1, min_child_samples=60,
        feature_fraction=0.75, bagging_fraction=0.85, bagging_freq=1,
        lambda_l1=0.5, lambda_l2=2.0, scale_pos_weight=spw,
        n_estimators=6000, random_state=seed, n_jobs=-1, verbosity=-1,
    )
    if LGB_DEVICE != "cpu":
        p.update(device=LGB_DEVICE, max_bin=255, gpu_use_dp=False)
    return p

# string copies for CatBoost 
Xtr_cb = X_train[FEATS].copy(); Xte_cb = X_test[FEATS].copy()
for c in CAT_COLS:
    Xtr_cb[c] = Xtr_cb[c].astype(str).fillna("NA")
    Xte_cb[c] = Xte_cb[c].astype(str).fillna("NA")


## 4b. Aligned out-of-fold predictions (ONE shared split) 

In [ ]:
# Same fold split for every model -> OOF columns line up -> the ensemble test in 4d is FAIR.
from scipy.stats import rankdata
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=42).split(X_train[FEATS], y))
MODEL_SEEDS = SEEDS  # 5-seed within-fold averaging (44,43,45,46,47)
lgb_models = []

def _cb_model(seed, use_gpu):
    kw = dict(task_type="GPU", devices="0") if use_gpu else {}
    return cb.CatBoostClassifier(
        loss_function="Logloss", eval_metric="AUC", learning_rate=0.03, depth=8,
        l2_leaf_reg=6, iterations=6000, random_seed=seed, scale_pos_weight=spw,
        od_wait=200, od_type="Iter", cat_features=CAT_COLS,
        verbose=False, allow_writing_files=False, **kw)

def _lgb_fp(s, tr, va):
    m = lgb.LGBMClassifier(**lgb_params(s))
    m.fit(X_train.iloc[tr][FEATS], y[tr], eval_set=[(X_train.iloc[va][FEATS], y[va])],
          eval_metric="auc", callbacks=[lgb.early_stopping(200, verbose=False)])
    lgb_models.append(m)
    return m.predict_proba(X_train.iloc[va][FEATS])[:,1], m.predict_proba(X_test[FEATS])[:,1]

def _cb_fp(s, tr, va):
    try:
        m = _cb_model(s, HAS_GPU); m.fit(Xtr_cb.iloc[tr], y[tr], eval_set=(Xtr_cb.iloc[va], y[va]))
    except Exception as e:
        if not HAS_GPU: raise
        print(f"    CatBoost GPU failed ({type(e).__name__}) -> CPU"); 
        m = _cb_model(s, False); m.fit(Xtr_cb.iloc[tr], y[tr], eval_set=(Xtr_cb.iloc[va], y[va]))
    return m.predict_proba(Xtr_cb.iloc[va])[:,1], m.predict_proba(Xte_cb)[:,1]

def run_oof(fp, name):
    oof = np.zeros(len(X_train)); pred = np.zeros(len(X_test))
    for k,(tr,va) in enumerate(FOLDS):
        pv = np.zeros(len(va)); pt = np.zeros(len(X_test))
        for s in MODEL_SEEDS:
            a,b = fp(s, tr, va); pv += a/len(MODEL_SEEDS); pt += b/len(MODEL_SEEDS)
        # per-fold rank-normalize -> removes cross-fold scale mismatch (the pooled-OOF bug)
        oof[va] = rankdata(pv) / len(pv)
        pred += (rankdata(pt) / len(pt)) / len(FOLDS)
        print(f"  {name} fold {k}: AUC={roc_auc_score(y[va],pv):.5f}")
    print(f"{name} OOF AUC = {roc_auc_score(y,oof):.5f}")
    return oof, pred

oof_lgb, pred_lgb = run_oof(_lgb_fp, "lgb"); gc.collect()
oof_cb,  pred_cb  = run_oof(_cb_fp,  "cb");  gc.collect()


## 4c. Neural net (quantile-normal transform) on the SAME folds 

In [ ]:
RUN_NN = True
oof_nn = pred_nn = None
if RUN_NN:
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models, callbacks
    except Exception:
        pipq("tensorflow"); import tensorflow as tf
        from tensorflow.keras import layers, models, callbacks
    from sklearn.preprocessing import QuantileTransformer

    num_feats = [c for c in FEATS if c not in CAT_COLS]
    Xtr_nn = X_train[num_feats].replace([np.inf,-np.inf], np.nan).fillna(0).values.astype("float32")
    Xte_nn = X_test[num_feats].replace([np.inf,-np.inf], np.nan).fillna(0).values.astype("float32")

    def build(n_in):
        m = models.Sequential([
            layers.Input((n_in,)),
            layers.Dense(256, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
            layers.Dense(128, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.2),
            layers.Dense(64, activation="relu"),
            layers.Dense(1, activation="sigmoid")])
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.AUC(name="auc")])
        return m

    from scipy.stats import rankdata
    # NN is the slow part: averaging over 2 of the 5 seeds keeps runtime sane.
    # Set NN_SEEDS = SEEDS for the full 5 (slower) if you have GPU time.
    NN_SEEDS = SEEDS[:2]
    cw = {0: 1.0, 1: float((y==0).sum()/max((y==1).sum(),1))}
    oof_nn = np.zeros(len(X_train)); pred_nn = np.zeros(len(X_test))
    for k,(tr,va) in enumerate(FOLDS):
        qt = QuantileTransformer(output_distribution="normal", subsample=200000,
                                 random_state=42).fit(Xtr_nn[tr])
        xa, xb, xt = qt.transform(Xtr_nn[tr]), qt.transform(Xtr_nn[va]), qt.transform(Xte_nn)
        pv = np.zeros(len(va)); pt = np.zeros(len(X_test))
        for s in NN_SEEDS:
            tf.keras.utils.set_random_seed(s)
            net = build(xa.shape[1])
            net.fit(xa, y[tr], validation_data=(xb, y[va]), epochs=60, batch_size=2048,
                    class_weight=cw, verbose=0,
                    callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max",
                               patience=8, restore_best_weights=True)])
            pv += net.predict(xb, batch_size=8192, verbose=0).ravel()/len(NN_SEEDS)
            pt += net.predict(xt, batch_size=8192, verbose=0).ravel()/len(NN_SEEDS)
        oof_nn[va] = rankdata(pv)/len(pv)              # per-fold rank-norm (same fix as trees)
        pred_nn += (rankdata(pt)/len(pt))/len(FOLDS)
        print(f"  nn fold {k}: AUC={roc_auc_score(y[va], pv):.5f}")
    print(f"nn OOF AUC = {roc_auc_score(y, oof_nn):.5f}")


## 4d. Is the ensemble helping or hurting? (auto-decided on OOF)

In [ ]:
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
def r01(a): return rankdata(a)/len(a)

oofs  = {"lgb":oof_lgb, "cb":oof_cb}
preds = {"lgb":pred_lgb, "cb":pred_cb}
if RUN_NN and oof_nn is not None:
    oofs["nn"]=oof_nn; preds["nn"]=pred_nn
names = list(oofs)

single = {n: roc_auc_score(y, oofs[n]) for n in names}
print("Per-model OOF AUC:")
for n in names: print(f"  {n:4s}: {single[n]:.5f}")
best_single = max(single, key=single.get)

Roof  = np.column_stack([r01(oofs[n])  for n in names])
Rpred = np.column_stack([r01(preds[n]) for n in names])
print("\nOOF rank-correlation (diversity check):")
print(pd.DataFrame(Roof, columns=names).corr().round(4).to_string())

auc_equal = roc_auc_score(y, Roof.mean(1))

rng = np.random.RandomState(0); best_w = None; auc_wopt = -1
for _ in range(5000):
    w = rng.dirichlet(np.ones(len(names)))
    a = roc_auc_score(y, Roof @ w)
    if a > auc_wopt: auc_wopt, best_w = a, w

stack_oof = cross_val_predict(LogisticRegression(max_iter=1000), Roof, y, cv=5,
                              method="predict_proba")[:,1]
auc_stack = roc_auc_score(y, stack_oof)

cands = {f"best single [{best_single}]": single[best_single],
         "equal blend": auc_equal, "optimized weights": auc_wopt,
         "logistic stacker": auc_stack}
print("\nEnsemble comparison (OOF AUC, higher = better):")
for k,v in sorted(cands.items(), key=lambda x:-x[1]): print(f"  {v:.5f}  {k}")

winner = max(cands, key=cands.get)
margin = cands[winner] - single[best_single]
print(f"\n=== VERDICT: '{winner}' wins by {margin:+.5f} over best single model ===")
if winner.startswith("best single"):
    print("-> The ensemble is NOT helping. Using the single best model.")
    oof_blend, pred_blend = oofs[best_single], preds[best_single]
elif winner == "equal blend":
    print("-> Equal blend best.")
    oof_blend, pred_blend = Roof.mean(1), Rpred.mean(1)
elif winner == "optimized weights":
    print("-> Weighted blend best. weights:", dict(zip(names, np.round(best_w,3))))
    oof_blend, pred_blend = Roof @ best_w, Rpred @ best_w
else:
    print("-> Logistic stacker best.")
    lr = LogisticRegression(max_iter=1000).fit(Roof, y)
    oof_blend, pred_blend = stack_oof, lr.predict_proba(Rpred)[:,1]

print(f"\nSelected OOF AUC = {roc_auc_score(y, oof_blend):.5f}")
k = int(0.10*len(y)); top = np.argsort(-oof_blend)[:k]
print(f"Precision@10% = {y[top].mean():.4f} | Recall@10% = {y[top].sum()/y.sum():.4f}")


## 5. Submission (probabilities — required by README, maximizes AUC) 

In [ ]:

sub = pd.DataFrame({"ACCOUNT_ID": X_test["acct"].values, "CHURN_PROB": pred_blend})
assert list(sub.columns) == ["ACCOUNT_ID", "CHURN_PROB"]
assert sub["ACCOUNT_ID"].is_unique and len(sub) == len(test_ids) and sub["CHURN_PROB"].notna().all()
assert sub["CHURN_PROB"].between(0, 1).all()
sub.to_csv("predictions.csv", index=False); sub.to_csv("submission.csv", index=False)
print(sub.head().to_string(index=False), f"\nSaved {len(sub)} rows (float probs in [0,1]).")


## 6. Explainability — SHAP

In [ ]:
RUN_SHAP = True
if RUN_SHAP:
    try:
        import shap
    except ImportError:
        pipq("shap"); import shap
    import matplotlib.pyplot as plt
    os.makedirs("explainability", exist_ok=True)
    sample = X_train[FEATS].sample(min(20000, len(X_train)), random_state=42)
    expl = shap.TreeExplainer(lgb_models[-1])
    sv = expl.shap_values(sample)
    sv = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv, sample, max_display=25, show=False)
    plt.tight_layout(); plt.savefig("explainability/shap_summary.png", dpi=150); plt.show()

    imp = (pd.DataFrame({"feature": FEATS,
                         "mean_abs_shap": np.abs(sv).mean(0)})
           .sort_values("mean_abs_shap", ascending=False))
    imp.to_csv("explainability/shap_importance.csv", index=False)
    print(imp.head(20).to_string(index=False))
